# GPU Kernel Generation — Kaggle Runner

This notebook sets up the environment to run the `llm-mlir-compiler` pipeline on Kaggle.

**What it does:**
1. Installs Ollama (Linux) and pulls `gemma4:e2b`
2. Installs Python dependencies
3. Extracts the MLIR Python bindings tarball
4. Runs the benchmark pipeline (`run_benchmarks.py`)

**Prerequisites:**
- Upload this repo as a Kaggle Dataset or `git clone` it in the first cell
- Turn on GPU acceleration in session settings (T4 x2 recommended)

In [ ]:
# Cell 1: Clone the repository (or skip if uploaded as dataset)
# If you uploaded the repo as a Kaggle Dataset, mount it instead.

import os

REPO_URL = "https://github.com/seradotcom/gpu-kernel-generation.git"  # Update if needed
REPO_DIR = "/kaggle/working/gpu-kernel-generation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls -la

In [ ]:
# Cell 2: Install Ollama and pull Gemma model

import subprocess
import time
import os

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
!nohup ollama serve > /tmp/ollama.log 2>&1 &
time.sleep(5)

# Pull the model
!ollama pull gemma4:e2b

print("Ollama ready. Model pulled.")

In [ ]:
# Cell 3: Install Python dependencies
!pip install -q -r requirements.txt
print("Python deps installed.")

In [ ]:
# Cell 4: Verify MLIR bindings extract correctly

import sys
sys.path.insert(0, "/kaggle/working/gpu-kernel-generation")

from core.config import MLIR_BINDINGS_PATH
print(f"MLIR bindings path: {MLIR_BINDINGS_PATH}")

try:
    from mlir.ir import Context
    print("✅ MLIR bindings import successful")
except ImportError as e:
    print(f"❌ MLIR import failed: {e}")

In [ ]:
# Cell 5: Create .env file for local Ollama

env_content = """
USE_REMOTE_MODEL=0
MODEL_OLLAMA=gemma4:e2b
OLLAMA_ENDPOINT=http://localhost:11434/api/chat
"""

with open("/kaggle/working/gpu-kernel-generation/.env", "w") as f:
    f.write(env_content)

print(".env created for Ollama local mode.")

In [ ]:
# Cell 6: Quick smoke test — check that the pipeline modules import

import sys
sys.path.insert(0, "/kaggle/working/gpu-kernel-generation")

from core.schemas import MlirResponse
from core.semantic_validator import SemanticValidator
from core.mlir_translator import MLIRTranslator
from core.triton_python_generator import TritonPythonGenerator
from core.triton_executor import TritonExecutor
from core.prompt_builder import PromptBuilder

print("✅ All core modules imported successfully.")

In [ ]:
# Cell 7: Run the benchmark pipeline
# This will generate JSON via Ollama -> verify MLIR -> generate Triton Python -> compile & benchmark

%cd /kaggle/working/gpu-kernel-generation
!python run_benchmarks.py

In [ ]:
# Cell 8 (Optional): Test a single custom prompt interactively

import json
import sys
sys.path.insert(0, "/kaggle/working/gpu-kernel-generation")

from core.llm_client import generate_llm_response
from core.schemas import MlirResponse
from core.semantic_validator import SemanticValidator
from core.mlir_translator import MLIRTranslator
from core.triton_python_generator import TritonPythonGenerator
from core.triton_executor import TritonExecutor
from core.prompt_builder import PromptBuilder

prompt_builder = PromptBuilder()
validator = SemanticValidator()
translator = MLIRTranslator()
gen = TritonPythonGenerator()
executor = TritonExecutor()

user_prompt = "Generate a Triton kernel that performs element-wise addition of two float32 vectors."
system_prompt = prompt_builder.build_prompt(user_prompt, MlirResponse.model_json_schema())

print("Calling Ollama...")
raw = generate_llm_response("ollama", system_prompt, user_prompt, schema=MlirResponse)
print("Raw response received.")

clean = raw.strip()
if clean.startswith("```json"): clean = clean[7:]
if clean.startswith("```"): clean = clean[3:]
if clean.endswith("```"): clean = clean[:-3]
clean = clean.strip()

mlir_obj = MlirResponse(**json.loads(clean))
errors = validator.validate(mlir_obj)
print(f"Semantic errors: {errors}")

mlir_code = translator.translate_to_module(mlir_obj.code)
print("MLIR verified OK")

triton_py = gen.generate(mlir_obj.code)
print("Generated Triton Python:")
print(triton_py)

result = executor.run(triton_py, n_elements=1024)
print(f"Execution result: {result}")

---

## Troubleshooting

**Ollama won't start:** Check `!cat /tmp/ollama.log` for errors.

**Model download fails:** Kaggle's disk is 20GB. Gemma 2B is ~2GB. Make sure you have space.

**MLIR bindings fail to import:** The tarball may have been compiled for a different glibc. Ask your teammate to re-compile on Kaggle's Ubuntu environment and replace the tarball.

**Triton not found:** `pip install triton` should work on Linux x86_64. If it fails on Kaggle, report the error.

**CUDA out of memory:** Reduce `n_elements` in `run_benchmarks.py` or the executor call.